# JPEG AI as a Threat to Deepfake Detection
**Computer Vision — Spring 2026**

> **Reference:** E. D. Cannas et al., "Is JPEG AI Going to Change Image Forensics?,"
> *2025 IEEE/CVF International Conference on Computer Vision Workshops (ICCVW)*,
> Honolulu, HI, USA, 2025, pp. 1575–1586.
> doi: [10.1109/ICCVW69036.2025.00167](https://doi.org/10.1109/ICCVW69036.2025.00167)

---

**Abstract:** JPEG AI is the first international standard for end-to-end learned image
compression. Unlike classical JPEG — which partitions images into 8×8 blocks and quantises
their Discrete Cosine Transform (DCT) coefficients — JPEG AI employs a fully convolutional
encoder–decoder trained end-to-end to minimise a rate–distortion objective. The encoder maps
the full image to a compact latent representation; the decoder reconstructs pixels from that
latent code. Because no block boundaries or quantisation grid are imposed, the codec
introduces a **fundamentally different statistical signature**: high-frequency detail is
smoothed by the neural decoder rather than truncated by hard quantisation, artefacts are
spatially correlated in a non-block pattern, and the real/fake spectral gap that classical
codecs preserve is erased.

Deepfake detectors exploit precisely these **low-level frequency cues**. During deepfake
generation (e.g., GAN-based face swapping), the synthesis process leaves characteristic
traces in the image statistics: upsampling grids from the generator's transposed convolutions
create periodic high-frequency peaks in the power spectrum; blending masks at face boundaries
introduce local discontinuities; and re-encoding the synthetic face with a source codec
(H.264 in FF++) embeds specific quantisation footprints in the DCT coefficient histogram.
Detectors trained on these cues learn to distinguish real from fake based on these
**compression artefacts embedded during the generation process** — not on the semantic
content of the face itself.

When JPEG AI re-compresses the image, its neural decoder overwrites these forensic traces
with statistically neutral textures, collapsing the real/fake gap that the detector relied
upon. This project investigates: (1) how much JPEG AI compression degrades detector
performance, (2) why it degrades (frequency-domain analysis), and (3) how to mitigate
the degradation.

---

Project structure (following course guidelines):
1. **Imports** — all required packages
2. **Globals** — project-wide constants and paths
3. **Utils** — helper functions (power spectrum, DCT histogram, evaluation, plotting)
4. **Data** — FaceForensics++ loading + JPEG AI compression pipeline
5. **Network** — deepfake detector definitions (ResNet50, EfficientNet-B4)
6. **Train** — fine-tuning loop + augmentation strategies
7. **Evaluation** — Phase 0–3: split, degradation curve, frequency analysis, mitigation

---
## 1. Imports

In [1]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import cv2
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve

# Project modules
sys.path.insert(0, str(Path.cwd()))
from src.compression.jpegai_codec import compress_dataset, BPP_LEVELS

_rocm = bool(getattr(torch.version, "hip", None))
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()} | ROCm: {_rocm}")


PyTorch 2.9.1+rocm6.3 | CUDA available: True | ROCm: True


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


---
## 2. Globals

In [2]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
# ROCm (AMD) and CUDA (NVIDIA) both surface as torch.cuda.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    _backend = "ROCm/AMD" if getattr(torch.version, "hip", None) else "CUDA/NVIDIA"
    print(f"Using device: cuda [{_backend}] — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("Using device: cpu")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path.cwd()
DATA_DIR      = ROOT / "data"
ORIGINAL_DIR  = DATA_DIR / "original"     # raw dataset images
COMPRESSED_DIR= DATA_DIR / "compressed"   # JPEG AI outputs
RESULTS_DIR   = ROOT / "results"
CHECKPOINTS_DIR = ROOT / "checkpoints"

for d in [DATA_DIR, ORIGINAL_DIR, COMPRESSED_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Compression settings ──────────────────────────────────────────────────────
# BPP levels for the degradation curve
BPP_LEVELS    = [0.1, 0.3, 0.5, 0.8, 1.0, 2.0]
JPEGAI_PROFILE= "base"

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LR            = 1e-4
IMG_SIZE      = 224

# ── Dataset split ─────────────────────────────────────────────────────────────
# Expected folder structure inside ORIGINAL_DIR:
#   original/
#     real/   ← genuine images   (label 0)
#     fake/   ← deepfake images  (label 1)
LABEL_MAP = {"real": 0, "fake": 1}


Using device: cuda [ROCm/AMD] — AMD Radeon Graphics


---
## 3. Utils

In [3]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def compute_power_spectrum(img_gray: np.ndarray) -> np.ndarray:
    """2D power spectrum (log magnitude) of a grayscale image."""
    f = np.fft.fft2(img_gray.astype(np.float32))
    fshift = np.fft.fftshift(f)
    return 20 * np.log(np.abs(fshift) + 1e-8)


def azimuthal_average(spectrum_2d: np.ndarray) -> np.ndarray:
    """Radial (azimuthal) average of a 2D spectrum → 1D frequency profile."""
    h, w = spectrum_2d.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
    max_r = min(cy, cx)
    profile = np.array([spectrum_2d[r == i].mean() for i in range(max_r)])
    return profile


def compute_dct_histogram(img_gray: np.ndarray, bins: int = 256) -> tuple:
    """DCT coefficient histogram of a grayscale image (block size 8×8)."""
    h, w = img_gray.shape
    h = (h // 8) * 8
    w = (w // 8) * 8
    img = img_gray[:h, :w].astype(np.float32)
    coeffs = []
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block = img[i:i+8, j:j+8]
            dct = cv2.dct(block)
            coeffs.append(dct.flatten())
    all_coeffs = np.concatenate(coeffs)
    hist, edges = np.histogram(all_coeffs, bins=bins, range=(-100, 100))
    return hist, edges


def evaluate_detector(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device = DEVICE,
) -> dict:
    """Run inference and return AUC, accuracy, F1."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)
    return {
        "auc":      roc_auc_score(all_labels, all_probs),
        "accuracy": accuracy_score(all_labels, preds),
        "f1":       f1_score(all_labels, preds),
        "labels":   np.array(all_labels),
        "probs":    np.array(all_probs),
    }


def plot_degradation_curve(results: dict, metric: str = "auc", title: str = "") -> None:
    """
    Plot detector metric vs BPP.
    results: {detector_name: {bpp: metric_value}}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    for detector_name, bpp_metrics in results.items():
        bpps = sorted(bpp_metrics.keys())
        vals = [bpp_metrics[b] for b in bpps]
        ax.plot(bpps, vals, marker="o", label=detector_name)
    ax.set_xlabel("BPP (bits per pixel)")
    ax.set_ylabel(metric.upper())
    ax.set_title(title or f"Detector {metric.upper()} vs JPEG AI BPP")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"degradation_{metric}.png", dpi=150)
    plt.show()

---
## 4. Data
### 4.1 Dataset class

### 4.0 FaceForensics++ — Dataset structure

FF++ was downloaded with compression level **c23** (H.264 videos), real sequences from **youtube**:

```
data/ff++/
  original_sequences/youtube/c23/videos/      ← real .mp4
  manipulated_sequences/Deepfakes/c23/videos/ ← fake .mp4
```

Frames were pre-extracted via `scripts/extract_frames.sh` (ffmpeg) into:

```
data/original/
  real/   ← PNG frames from youtube videos   (label 0)
  fake/   ← PNG frames from Deepfakes videos (label 1)
```

The cell below verifies that everything is in place.

In [4]:
from pathlib import Path

# ── FF++ source video paths (c23, already downloaded) ────────────────────────
FF_ROOT  = DATA_DIR / "ff++"
real_src = FF_ROOT / "original_sequences" / "youtube" / "c23" / "videos"
fake_src = FF_ROOT / "manipulated_sequences" / "Deepfakes" / "c23" / "videos"

for label, src in [("real", real_src), ("fake", fake_src)]:
    if src.exists():
        n = len(list(src.glob("*.mp4")))
        print(f"FF++ {label} videos : {n}  ({src})")
    else:
        print(f"FF++ {label} source NOT found: {src}")

# ── Pre-extracted frames (output of scripts/extract_frames.sh) ───────────────
OUT_REAL = ORIGINAL_DIR / "real"
OUT_FAKE = ORIGINAL_DIR / "fake"

real_n = len(list(OUT_REAL.rglob("*.png"))) + len(list(OUT_REAL.rglob("*.jpg"))) if OUT_REAL.exists() else 0
fake_n = len(list(OUT_FAKE.rglob("*.png"))) + len(list(OUT_FAKE.rglob("*.jpg"))) if OUT_FAKE.exists() else 0

print(f"\nExtracted frames in data/original/")
print(f"  real : {real_n}")
print(f"  fake : {fake_n}")

if real_n == 0 or fake_n == 0:
    print("\n⚠  No frames found — run frame extraction first:")
    print("   bash scripts/extract_frames.sh data/ff++ data/original")
else:
    print("\n✓ Dataset ready.")


FF++ real videos : 200  (/home/cascatetto/ssd/Projects/jpegai-deepfake-forensics/jpegai-deepfake-forensics/data/ff++/original_sequences/youtube/c23/videos)
FF++ fake videos : 200  (/home/cascatetto/ssd/Projects/jpegai-deepfake-forensics/jpegai-deepfake-forensics/data/ff++/manipulated_sequences/Deepfakes/c23/videos)

Extracted frames in data/original/
  real : 10420
  fake : 10420

✓ Dataset ready.


In [5]:
class DeepfakeDataset(Dataset):
    """
    Loads images from a directory with 'real/' and 'fake/' subdirectories.
    Works for both original and JPEG-AI-compressed splits.
    """
    def __init__(self, root: Path, transform=None, label_map: dict = LABEL_MAP):
        self.samples = []
        for cls_name, label in label_map.items():
            cls_dir = root / cls_name
            if not cls_dir.exists():
                continue
            for p in sorted(cls_dir.rglob("*.png")) + sorted(cls_dir.rglob("*.jpg")):
                self.samples.append((p, label))
        self.transform = transform or T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def make_dataloader(root: Path, batch_size: int = BATCH_SIZE, shuffle: bool = False) -> DataLoader:
    ds = DeepfakeDataset(root)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=4, pin_memory=True)


# Verify dataset structure
print("Original dataset:")
for cls in ["real", "fake"]:
    cls_dir = ORIGINAL_DIR / cls
    count = len(list(cls_dir.rglob("*.png"))) + len(list(cls_dir.rglob("*.jpg"))) if cls_dir.exists() else 0
    print(f"  {cls}: {count} images")

Original dataset:
  real: 10420 images
  fake: 10420 images


### 4.2 JPEG AI Compression Pipeline

> **Run this step inside the `jpeg_ai_vm` conda env (Linux native).**  
> The compressed images are saved to `data/compressed/` and reused in all subsequent cells.  
> Skip this cell if `data/compressed/` already exists.

In [6]:
# ── Option A: Python API (if running inside jpeg_ai_vm env) ──────────────────
# compress_dataset(
#     dataset_dir=ORIGINAL_DIR,
#     output_root=COMPRESSED_DIR,
#     bpp_levels=BPP_LEVELS,
#     profile=JPEGAI_PROFILE,
# )

# ── Option B: Shell script (Linux native) ─────────────────────────────────────
# Run in terminal:
#   conda activate jpeg_ai_vm
#   bash scripts/compress_dataset.sh data/original data/compressed

# ── Verify compressed data exists ────────────────────────────────────────────
for bpp in BPP_LEVELS:
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / bpp_tag
    count = len(list(bpp_dir.rglob("*.png"))) if bpp_dir.exists() else 0
    status = "OK" if count > 0 else "MISSING"
    print(f"  [{status}] {bpp_tag}: {count} images")


  [OK] bpp_010: 1000 images
  [OK] bpp_030: 1000 images
  [OK] bpp_050: 1000 images
  [OK] bpp_080: 1000 images
  [OK] bpp_100: 1000 images
  [OK] bpp_200: 1000 images


---
## 5. Network
### 5.1 Detector definitions

In [7]:
import timm

def load_detector(name: str, pretrained: bool = True, num_classes: int = 2) -> nn.Module:
    """
    Load a deepfake detector backbone.

    Supported names:
        'resnet50'        — CNNDetection-style (Wang et al., CVPR 2020)
        'efficientnet_b4' — EfficientNet (timm)
        'xception'        — FaceForensics++ baseline
    """
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    return model.to(DEVICE)


# TODO: load pre-trained forensics weights if available.
# Example: CNNDetection weights from https://github.com/peterwang512/CNNDetection
#
# detector = load_detector('resnet50', pretrained=False)
# ckpt = torch.load('checkpoints/cnndetection_resnet50.pth', map_location=DEVICE)
# detector.load_state_dict(ckpt)

detector_names = ["resnet50", "efficientnet_b4"]
print("Detectors:", detector_names)

Detectors: ['resnet50', 'efficientnet_b4']


---
## 6. Train
### 6.1 Fine-tuning loop (Mitigation Strategy 1: compression-augmented training)

In [8]:

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def finetune(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = NUM_EPOCHS,
    lr: float = LR,
    save_path: Path = None,
    label: str = "",
) -> dict:
    """Fine-tune detector. Returns history dict with train_loss and val_auc per epoch."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_loss": [], "val_auc": []}

    best_auc = 0.0
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        metrics = evaluate_detector(model, val_loader)
        scheduler.step()
        history["train_loss"].append(train_loss)
        history["val_auc"].append(metrics["auc"])
        marker = " ◀ best" if metrics["auc"] > best_auc else ""
        best_auc = max(best_auc, metrics["auc"])
        print(f"  [{label}] Epoch {epoch:02d} | loss {train_loss:.4f} | val AUC {metrics['auc']:.4f}{marker}")

    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f"  Checkpoint saved: {save_path}")
    return history


def plot_training_curves(histories: dict) -> None:
    """Plot train loss and val AUC per epoch for each model."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax1.plot(h["train_loss"], marker="o", label=name)
        ax2.plot(h["val_auc"],   marker="o", label=name)
    ax1.set_title("Training Loss per Epoch")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.set_title("Validation AUC per Epoch")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("AUC")
    ax2.set_ylim(0.5, 1.02)
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
    plt.show()
    print(f"Plot saved → {RESULTS_DIR / 'training_curves.png'}")


# ── Compression-augmented dataset for fine-tuning ────────────────────────────
class AugmentedDeepfakeDataset(Dataset):
    """
    Mixes original + JPEG-AI-compressed images for robust fine-tuning.
    Randomly picks original or one compressed variant per sample each epoch.
    """
    def __init__(self, original_root: Path, compressed_root: Path,
                 bpp_levels: list = BPP_LEVELS, transform=None):
        self.base = DeepfakeDataset(original_root, transform)
        self.compressed_roots = [
            compressed_root / f"bpp_{int(b*100):03d}" for b in bpp_levels
        ]
        self.transform = self.base.transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        orig_path, label = self.base.samples[idx]
        candidates = [orig_path]
        for cr in self.compressed_roots:
            compressed_dir = cr / orig_path.parent.name
            matches = list(compressed_dir.glob(f"{orig_path.stem}_bpp*.png"))
            if matches:
                candidates.append(matches[0])
        chosen = random.choice(candidates)
        img = Image.open(chosen).convert("RGB")
        return self.transform(img), label


print("Train module ready.")


Train module ready.


---
## 7. Evaluation
### 7.1 Phase 0 — Data Split & Detector Fine-tuning

**Goal:** Build a leakage-free data partition and fine-tune two deepfake detectors.

#### Why the split design matters
FaceForensics++ frames extracted from the same source video are visually near-identical.
If a frame used for evaluation also appears (or its uncompressed original appears) in
the training set, the model can "memorise" it — inflating AUC artificially.

We enforce strict isolation with a **four-way partition**:

| Set | Source | Size | Role |
|-----|--------|------|------|
| `COMP_TRAIN` | First 250/class of compressed pool | 250 original stems/cls → **1 500 compressed images** (250 × 6 BPP) | MIT-1 augmented training only |
| `COMP_TEST` | Last 250/class of compressed pool | 250 original stems/cls → **1 500 compressed images** (250 × 6 BPP) | **All** compressed evaluation |
| `ORIG_TRAIN` | Remaining originals, 80% split | ~7,936/cls | Baseline + MIT fine-tuning |
| `ORIG_TEST` | Remaining originals, 20% split | ~1,984/cls | Evaluation on originals |

> **Why 250+250 and not 500 for evaluation?**
> We have **500 unique original images per class** that were compressed at all 6 BPP levels.
> We use **all 500** — but split them to prevent data leakage:
> - The 250 in `COMP_TRAIN` are used (as compressed) to train MIT-1.
>   They **cannot** also be used for evaluation, since MIT-1 has seen them.
> - The 250 in `COMP_TEST` are **never touched during any training phase**,
>   so evaluation on them is unbiased.
> 
> In total: `COMP_TRAIN` contributes 250 × 6 = **1 500 compressed samples** to MIT-1 training.
> `COMP_TEST` contributes 250 × 6 = **1 500 compressed samples** for evaluation at each BPP.

`COMP_TEST` images are **never seen during training** in any form.
`ORIG_TRAIN`/`ORIG_TEST` images have **no compressed counterpart** in any split.

#### Two detectors — why?
The PDF specification asks for *"one or more pre-trained deepfake detectors"*.
We fine-tune two architectures to verify that JPEG AI degradation is
**backbone-agnostic** (not specific to one architecture):

---

**Detector 1 — ResNet50**

Introduced by He et al. (CVPR 2016), ResNet50 uses **residual (skip) connections**
to solve the vanishing-gradient problem in deep networks. The identity shortcut
lets gradients flow directly through the network, enabling 50 layers to be trained
stably. The architecture is organised in 4 stages of bottleneck blocks
(1×1 → 3×3 → 1×1 convolutions) that progressively downsample spatial resolution
while expanding channel depth (64 → 512 channels).

For deepfake detection, this backbone follows the **CNNDetection** setup
(Wang et al., CVPR 2020): the ImageNet classifier head is replaced with a
2-class linear layer, and the whole network is fine-tuned end-to-end.
CNNDetection showed that a standard ResNet50 trained with mild augmentation
generalises surprisingly well across unseen GAN architectures — the key signal
lies in low-level texture statistics rather than semantic content.

---

**Detector 2 — EfficientNet-B4**

Introduced by Tan & Le (ICML 2019), EfficientNet uses **compound scaling**:
rather than independently scaling depth, width, or resolution, a single
coefficient $\phi$ scales all three dimensions simultaneously via
$\text{depth} \propto \alpha^\phi$, $\text{width} \propto \beta^\phi$,
$\text{resolution} \propto \gamma^\phi$ (with $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$).
B4 sits at $\phi=4$: input resolution 380×380, ~19M parameters.
The building block is **MBConv** (mobile inverted bottleneck with depthwise separable
convolutions) with **Squeeze-and-Excitation** channel attention — very different
feature hierarchies from ResNet's plain residual stack.

Using EfficientNet-B4 alongside ResNet50 tests whether the JPEG AI degradation
effect is tied to a specific inductive bias (residual shortcuts vs. compound scaling)
or is a general forensic phenomenon.

---

**Training setup — shared by both detectors**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Initialisation** | ImageNet pretrained weights | Transfer learning from 1.2M images; rich low-level texture features available from epoch 1 |
| **Optimiser** | **AdamW** (lr=1e-4, weight\_decay=1e-4) | Adam with *decoupled* L2 regularisation (Loshchilov & Hutter, ICLR 2019). Standard Adam folds weight decay into the gradient update, inadvertently scaling it by the adaptive learning rate; AdamW applies it directly to weights — better regularisation and more stable fine-tuning |
| **LR schedule** | **Cosine annealing** (`CosineAnnealingLR`, T\_max=epochs) | Smoothly decays the learning rate from `lr` → 0 following a half-cosine curve. Avoids the abrupt drops of step schedules; the model can escape sharp minima early and converges to flatter, better-generalising minima near the end |
| **Loss** | Cross-entropy (binary: real=0, fake=1) | Standard classification objective; numerically stable with softmax outputs |
| **Epochs** | 25 | Sufficient for convergence on ~7 936 samples/class; monitored via val AUC to detect overfitting |
| **Batch size** | 32 | Fits in GPU memory; large enough for stable gradient estimates |

### 7.1b Phase 1 — Baseline Degradation Curve

**Goal:** Quantify how much JPEG AI compression degrades detector performance across
a range of bitrates, using both trained detectors as baselines.

#### What we measure

For each detector × condition we compute three metrics:

| Metric | Definition | Why it matters |
|--------|-----------|----------------|
| **AUC** (primary) | Area under the ROC curve | Threshold-independent; summarises separability between real and fake across all operating points. AUC=1.0 is perfect, AUC=0.5 is chance |
| **Accuracy** | Fraction correctly classified at threshold 0.5 | Intuitive, but sensitive to class balance |
| **F1** | Harmonic mean of precision and recall | Better than accuracy when class distributions are unequal |

AUC is the primary metric because it does not depend on a fixed decision threshold — a
detector that ranks fakes above reals is useful regardless of what threshold is applied.

#### Evaluation sets

We evaluate on **two distinct pools** per condition:

- **`ORIG_TEST`** (uncompressed, ~1 984/class): measures the detector's ability on
  images it has never seen but that match the training distribution (original FF++ frames).
  This is the **upper-bound** — the best the detector can do with no codec interference.

- **`COMP_TEST`** at each of 6 BPP levels (250 compressed images/class per BPP):
  measures how performance degrades as more aggressive neural compression is applied.
  Lower BPP = more compression = more forensic-cue destruction.

  | BPP | Bits per pixel | Approximate compression ratio vs raw |
  |-----|---------------|---------------------------------------|
  | 0.1 | very low | ~240× |
  | 0.3 | low | ~80× |
  | 0.5 | moderate | ~48× |
  | 0.8 | medium | ~30× |
  | 1.0 | medium-high | ~24× |
  | 2.0 | high | ~12× |

#### What to look for in the results

- **AUC drops monotonically with lower BPP**: JPEG AI destroys the high-frequency
  forensic cues (GAN upsampling grids, blending artefacts) that the detector exploits.
- **Both detectors degrade similarly**: confirms the effect is backbone-agnostic — it
  is a property of the codec, not of a specific architecture.
- **The gap between `ORIG_TEST` AUC and `COMP_TEST` AUC at each BPP** measures the
  absolute degradation; this gap motivates the mitigation strategies in Phase 3.

Results are saved to `results/degradation_results.csv` for reproducibility.

In [ ]:

# ── Phase 0: Clean train/test split — no data leakage ────────────────────────
#
# Split layout (per class):
#   Compressed pool  → 500 originals (identified by stem) + their 6×BPP PNG
#       └─ COMP_TRAIN  : first 250  → used in MIT-1 augmented training
#       └─ COMP_TEST   : last  250  → used in ALL evaluation on compressed sets
#   Remaining originals (~9,920) → 80/20 split
#       └─ ORIG_TRAIN  : 80% → baseline + MIT fine-tuning
#       └─ ORIG_TEST   : 20% → baseline + MIT evaluation on originals

from sklearn.model_selection import train_test_split as _tts
import re

_base_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def _compressed_stems(bpp: float, cls: str) -> set:
    """Return the set of original image stems that have a compressed version at this BPP."""
    d = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}" / cls
    stems = set()
    for p in d.rglob("*.png"):
        stems.add(re.sub(r"_bpp\d+$", "", p.stem))
    return stems


# Collect all stems that appear in any BPP directory (union across all BPP levels)
_comp_stems_real = set()
_comp_stems_fake = set()
for _bpp in BPP_LEVELS:
    _comp_stems_real |= _compressed_stems(_bpp, "real")
    _comp_stems_fake |= _compressed_stems(_bpp, "fake")

print(f"Compressed originals — real: {len(_comp_stems_real)}, fake: {len(_comp_stems_fake)}")

_all_real = sorted((ORIGINAL_DIR / "real").rglob("*.png")) + \
            sorted((ORIGINAL_DIR / "real").rglob("*.jpg"))
_all_fake = sorted((ORIGINAL_DIR / "fake").rglob("*.png")) + \
            sorted((ORIGINAL_DIR / "fake").rglob("*.jpg"))

# Split into: images that have a compressed version vs. images that don't
_comp_real = sorted([p for p in _all_real if p.stem in _comp_stems_real])
_comp_fake = sorted([p for p in _all_fake if p.stem in _comp_stems_fake])
_free_real  = [p for p in _all_real if p.stem not in _comp_stems_real]
_free_fake  = [p for p in _all_fake if p.stem not in _comp_stems_fake]

# COMP pool: 50/50 split → COMP_TRAIN (MIT-1 only) | COMP_TEST (all evals)
_n_comp = len(_comp_real)
_half   = _n_comp // 2
COMP_TRAIN_STEMS_REAL = {p.stem for p in _comp_real[:_half]}
COMP_TEST_STEMS_REAL  = {p.stem for p in _comp_real[_half:]}
COMP_TRAIN_STEMS_FAKE = {p.stem for p in _comp_fake[:_half]}
COMP_TEST_STEMS_FAKE  = {p.stem for p in _comp_fake[_half:]}

# Free originals: 80/20 train/test split (stratified by random_state for reproducibility)
_free_real_tr, _free_real_te = _tts(_free_real, test_size=0.2, random_state=SEED)
_free_fake_tr, _free_fake_te = _tts(_free_fake, test_size=0.2, random_state=SEED)

ORIG_TRAIN = [(p, 0) for p in _free_real_tr] + [(p, 1) for p in _free_fake_tr]
ORIG_TEST  = [(p, 0) for p in _free_real_te] + [(p, 1) for p in _free_fake_te]
TRAIN_SAMPLES = ORIG_TRAIN
TEST_SAMPLES  = ORIG_TEST

print(f"\nSplit summary:")
print(f"  ORIG_TRAIN : {len(ORIG_TRAIN):>6}  ({len(_free_real_tr)} real, {len(_free_fake_tr)} fake)")
print(f"  ORIG_TEST  : {len(ORIG_TEST):>6}  ({len(_free_real_te)} real, {len(_free_fake_te)} fake)")
print(f"  COMP_TRAIN : {_half}/class = {_half*2} total — used in MIT-1 training")
print(f"  COMP_TEST  : {_n_comp - _half}/class = {(_n_comp-_half)*2} total — used in all compressed evals")
print(f"\n  Leakage check: COMP_TEST ∩ ORIG_TRAIN = {len(COMP_TEST_STEMS_REAL & {p.stem for p,_ in ORIG_TRAIN})} (must be 0)")


def make_list_loader(samples, shuffle=False, transform=None):
    class _LD(Dataset):
        def __init__(self, s, t): self.s, self.t = s, t
        def __len__(self): return len(self.s)
        def __getitem__(self, i):
            p, l = self.s[i]
            return (self.t or _base_transform)(Image.open(p).convert("RGB")), l
    return DataLoader(_LD(samples, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=4, pin_memory=True)


def make_comp_test_loader(bpp: float) -> DataLoader:
    """Load COMP_TEST compressed images at a given BPP (used for evaluation only)."""
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    samples = []
    for p in sorted((bpp_dir / "real").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_REAL:
            samples.append((p, 0))
    for p in sorted((bpp_dir / "fake").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_FAKE:
            samples.append((p, 1))
    return make_list_loader(samples)


def make_comp_train_samples(bpp: float) -> list:
    """Return COMP_TRAIN compressed image paths at a given BPP (used for MIT-1 training)."""
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    samples = []
    for p in sorted((bpp_dir / "real").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TRAIN_STEMS_REAL:
            samples.append((p, 0))
    for p in sorted((bpp_dir / "fake").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TRAIN_STEMS_FAKE:
            samples.append((p, 1))
    return samples


# ── Fine-tune Detector 1: ResNet50 ───────────────────────────────────────────
# ResNet50 follows the CNNDetection setup (Wang et al., CVPR 2020).
# We start from ImageNet weights and fine-tune only on ORIG_TRAIN —
# images with no compressed counterpart, so the model learns genuine
# real-vs-fake artefacts without any JPEG AI codec exposure.
BASELINE_DET    = "resnet50"
BASELINE_EPOCHS = 25
baseline_ckpt   = CHECKPOINTS_DIR / f"{BASELINE_DET}_baseline.pth"
_training_histories = {}

if baseline_ckpt.exists():
    print(f"\nCheckpoint found — skipping training: {baseline_ckpt}")
else:
    print(f"\nFine-tuning {BASELINE_DET} for {BASELINE_EPOCHS} epochs on ORIG_TRAIN...")
    _m = load_detector(BASELINE_DET)
    _training_histories[f"Baseline ({BASELINE_DET})"] = finetune(
        _m,
        make_list_loader(ORIG_TRAIN, shuffle=True),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=baseline_ckpt,
        label="baseline",
    )

baseline_model = load_detector(BASELINE_DET, pretrained=False)
baseline_model.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE, weights_only=True))
baseline_model.eval()
_m = evaluate_detector(baseline_model, make_list_loader(ORIG_TEST))
print(f"\nBaseline {BASELINE_DET} on ORIG_TEST — AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")


# ── Fine-tune Detector 2: EfficientNet-B4 ────────────────────────────────────
# EfficientNet-B4 uses compound scaling (balancing depth/width/resolution).
# Including a second backbone verifies whether the JPEG AI degradation
# effect is architecture-agnostic or specific to ResNet-style features.
SECOND_DET  = "efficientnet_b4"
second_ckpt = CHECKPOINTS_DIR / f"{SECOND_DET}_baseline.pth"

if second_ckpt.exists():
    print(f"\nCheckpoint found — skipping training: {second_ckpt}")
else:
    print(f"\nFine-tuning {SECOND_DET} for {BASELINE_EPOCHS} epochs on ORIG_TRAIN...")
    _m2 = load_detector(SECOND_DET)
    _training_histories[f"Baseline ({SECOND_DET})"] = finetune(
        _m2,
        make_list_loader(ORIG_TRAIN, shuffle=True),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=second_ckpt,
        label=SECOND_DET,
    )

second_model = load_detector(SECOND_DET, pretrained=False)
second_model.load_state_dict(torch.load(second_ckpt, map_location=DEVICE, weights_only=True))
second_model.eval()
_m2 = evaluate_detector(second_model, make_list_loader(ORIG_TEST))
print(f"\nBaseline {SECOND_DET} on ORIG_TEST — AUC={_m2['auc']:.4f}  acc={_m2['accuracy']:.4f}")


In [ ]:

# Phase 1: Evaluate both fine-tuned detectors on ORIG_TEST + COMP_TEST.
#
# For each detector we measure AUC, accuracy and F1 at each BPP level,
# building the degradation curve: performance vs. compression strength.
#
# COMP_TEST is strictly disjoint from any training data (no leakage).
# Evaluating two architectures lets us check whether the degradation is
# backbone-agnostic — if both drop similarly, it is a general phenomenon.

phase1_results        = {}   # ResNet50
phase1_results_second = {}   # EfficientNet-B4in this cell

# ── ResNet50 ──────────────────────────────────────────────────────────────────
print(f"=== {BASELINE_DET} ===")
_m = evaluate_detector(baseline_model, make_list_loader(ORIG_TEST))
phase1_results["original"] = _m
print(f"  original (ORIG_TEST) | AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")

for bpp in tqdm(BPP_LEVELS, desc=f"{BASELINE_DET} @ compressed"):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    if not bpp_dir.exists():
        print(f"  SKIP {bpp_tag}")
        continue
    _m = evaluate_detector(baseline_model, make_comp_test_loader(bpp))
    phase1_results[bpp] = _m
    print(f"  {bpp_tag} (COMP_TEST) | AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")

# ── EfficientNet-B4 ──────────────────────────────────────────────────────────
print(f"\n=== {SECOND_DET} ===")
_m = evaluate_detector(second_model, make_list_loader(ORIG_TEST))
phase1_results_second["original"] = _m
print(f"  original (ORIG_TEST) | AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")

for bpp in tqdm(BPP_LEVELS, desc=f"{SECOND_DET} @ compressed"):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    if not bpp_dir.exists():
        continue
    _m = evaluate_detector(second_model, make_comp_test_loader(bpp))
    phase1_results_second[bpp] = _m
    print(f"  {bpp_tag} (COMP_TEST) | AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")

# ── Save CSV ──────────────────────────────────────────────────────────────────
rows = []
for det, res in [(BASELINE_DET, phase1_results), (SECOND_DET, phase1_results_second)]:
    for k, v in res.items():
        rows.append({"detector": det, "strategy": "baseline", "bpp": k,
                     "auc": v["auc"], "accuracy": v["accuracy"], "f1": v["f1"]})
pd.DataFrame(rows).to_csv(RESULTS_DIR / "degradation_results.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'degradation_results.csv'}")

# ── Degradation curve — both detectors on the same plot ──────────────────────
# A drop in AUC from "original" → low BPP means JPEG AI disrupts the
# artefact patterns that the detector relies upon.
_auc_both = {
    f"Baseline ({BASELINE_DET})": {k: v["auc"] for k, v in phase1_results.items() if k != "original"},
    f"Baseline ({SECOND_DET})":   {k: v["auc"] for k, v in phase1_results_second.items() if k != "original"},
}
plot_degradation_curve(_auc_both, metric="auc",
                       title="Phase 1 — Detector AUC vs JPEG AI BPP")
print(f"Plot saved → {RESULTS_DIR / 'degradation_auc.png'}")


### 7.2 Phase 2 — Frequency-Domain Forensic Analysis

**Goal:** Explain *why* detector performance degrades under JPEG AI compression.

Deepfake detectors exploit **low-level frequency artefacts** left by the
generation process — e.g., GAN upsampling grids, blending boundary noise,
or specific quantisation patterns from the source codec. JPEG AI, being a
*neural codec*, introduces fundamentally different artefacts from classical JPEG.

We visualise this through two analyses:

**1 — Azimuthal power spectra (this cell)**
The radially-averaged 2D FFT magnitude gives a 1D profile of how image
energy is distributed across spatial frequencies (low = coarse structure,
high = fine texture/artefacts).

- Uncompressed images show a characteristic $1/f$ roll-off with bumps at
  frequencies corresponding to deepfake generation artefacts.
- After JPEG AI compression, high-frequency energy is attenuated and the
  real/fake spectral gap narrows — the cues that the detector relied on are
  washed out by the neural codec.

**2 — DCT coefficient histograms (next cell)**
The 8×8 block DCT is the foundation of classical JPEG and is widely exploited
by forensic tools. JPEG AI does not use block-DCT internally, so it distorts
the coefficient histogram in a different way — explaining why detectors
trained without compression fail to generalise.

In [ ]:

# Compare power spectra across: original-real, original-fake, compressed-real, compressed-fake

N_SAMPLES = 100  # images per category

def collect_spectra(img_dir: Path, label: str, n: int = N_SAMPLES) -> np.ndarray:
    paths = list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg"))
    paths = sorted(paths)[:n]
    spectra = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        spectra.append(azimuthal_average(compute_power_spectrum(gray)))
    # Truncate all profiles to the minimum length (images may have different sizes)
    min_len = min(len(s) for s in spectra)
    return np.stack([s[:min_len] for s in spectra])  # shape (n, min_len)


fig, axes = plt.subplots(1, len(BPP_LEVELS) + 1, figsize=(20, 4), sharey=True)

# Original
for cls in ["real", "fake"]:
    specs = collect_spectra(ORIGINAL_DIR / cls, cls)
    axes[0].plot(specs.mean(axis=0), label=cls)
axes[0].set_title("Original")
axes[0].set_xlabel("Frequency (px⁻¹)")
axes[0].set_ylabel("Power (dB)")
axes[0].legend()

# Compressed at each BPP
for i, bpp in enumerate(BPP_LEVELS):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    ax = axes[i + 1]
    for cls in ["real", "fake"]:
        bpp_cls_dir = COMPRESSED_DIR / bpp_tag / cls
        if not bpp_cls_dir.exists():
            continue
        specs = collect_spectra(bpp_cls_dir, cls)
        ax.plot(specs.mean(axis=0), label=cls)
    ax.set_title(f"BPP={bpp}")
    ax.set_xlabel("Frequency (px⁻¹)")
    ax.legend()

plt.suptitle("Mean Azimuthal Power Spectra — JPEG AI Compression Effect", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "power_spectra.png", dpi=150)
plt.show()


### 7.3 Phase 2b — DCT Coefficient Distributions

#### What is the DCT?

The **Discrete Cosine Transform** decomposes a signal into a sum of cosine functions
oscillating at different frequencies. For an 8×8 block of pixel values $f(x,y)$, the
2D DCT produces 64 coefficients $F(u,v)$:

$$F(u,v) = \frac{2}{N} C(u) C(v) \sum_{x=0}^{N-1} \sum_{y=0}^{N-1}
f(x,y) \cos\!\left[\frac{\pi(2x+1)u}{2N}\right] \cos\!\left[\frac{\pi(2y+1)v}{2N}\right]$$

where $N=8$, and $C(k) = 1/\sqrt{2}$ for $k=0$, else $C(k)=1$.

- **$F(0,0)$** is the DC coefficient — the mean pixel intensity of the block (low frequency).
- **$F(u,v)$ for large $u,v$** are AC coefficients — they encode fine horizontal/vertical
  texture (high frequency).

#### Why 8×8 blocks?

Classical JPEG partitions the image into non-overlapping 8×8 pixel blocks before applying the
DCT. This block size was chosen in the 1992 JPEG standard as a trade-off between:
- **compression efficiency** (larger blocks capture more redundancy)
- **blocking artefact visibility** (larger blocks create coarser artefacts when quantised)

Because JPEG has been the dominant codec for 30+ years, forensic tools and deepfake
detectors have learned to exploit the statistical patterns these 8×8 blocks leave in images.

#### The histogram

We flatten all DCT coefficients from all 8×8 blocks of an image into a 1D array and
compute a histogram over the range [−100, 100]. This gives a **global frequency fingerprint**
of the image.

**What the histogram shape reveals:**

| Pattern | Cause | Forensic implication |
|---------|-------|----------------------|
| **Sharp Laplacian (double-exponential) peak at 0** | Natural images: most 8×8 blocks are spatially smooth, so high-frequency AC coefficients are near zero | Expected in genuine photos |
| **Wider, flatter distribution** | Compression or generation artefacts spread energy into higher-frequency coefficients | Indicates the image has been processed |
| **Real vs. fake gap in originals** | GAN generators introduce characteristic high-frequency artefacts (upsampling grids, blending edges) absent in camera images | The signal deepfake detectors exploit |
| **Real ≈ fake after JPEG AI compression** | The neural codec's decoder smooths both real and fake images similarly, collapsing the gap | Explains why detector AUC drops in Phase 1 |

#### JPEG AI vs classical JPEG — a key difference

Classical JPEG quantises DCT coefficients directly: it divides each $F(u,v)$ by a
quantisation step $Q(u,v)$ and rounds to an integer. This forces AC coefficients to land on
a discrete grid — a forensic "grid artefact" visible in the histogram as periodic spikes.

JPEG AI does **not** use block-DCT internally. Its encoder is a CNN that maps the full
image to a compact latent representation; the decoder reconstructs pixels from that
latent code. The result:

- No block boundaries → no 8×8 tiling artefacts
- No quantisation grid → histogram spikes disappear
- Decoder hallucination smooths fine detail → AC coefficients near-zero across both real and fake

This is why JPEG AI is more forensically disruptive than classical JPEG at equivalent
file size: it does not just remove high-frequency information — it replaces it with
statistically neutral textures that detectors cannot distinguish.

We compute histograms over **50 images per category per BPP level**, using all 8×8 blocks
in each image, then plot the mean normalised histogram.

**What to look for in the plots:**

| Observation | Interpretation |
|-------------|----------------|
| Real vs. fake diverge at original | Detector has a signal to exploit |
| Gap narrows as BPP decreases | JPEG AI erases the forensic fingerprint |
| Histogram widens under compression | Neural decoder spreads coefficient energy |
| At BPP=0.1, real ≈ fake | Detection approaches chance — motivates mitigation |

In [ ]:
fig, axes = plt.subplots(2, len(BPP_LEVELS) + 1, figsize=(22, 7))

def plot_dct_hist(ax, img_dir: Path, label: str, color: str, n: int = 50) -> None:
    paths = (list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg")))[:n]
    all_hists = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        hist, edges = compute_dct_histogram(gray)
        all_hists.append(hist / hist.sum())  # normalize
    centers = (edges[:-1] + edges[1:]) / 2
    mean_hist = np.mean(all_hists, axis=0)
    ax.plot(centers, mean_hist, color=color, label=label, linewidth=1.5)
    ax.set_xlim(-60, 60)
    ax.set_xlabel("DCT coefficient value")
    ax.set_ylabel("Normalized count")
    ax.legend(fontsize=8)

for row, cls in enumerate(["real", "fake"]):
    color = "steelblue" if cls == "real" else "tomato"
    # Original
    plot_dct_hist(axes[row][0], ORIGINAL_DIR / cls, f"original/{cls}", color)
    axes[row][0].set_title(f"Original – {cls}")
    # Compressed
    for i, bpp in enumerate(BPP_LEVELS):
        bpp_tag = f"bpp_{int(bpp*100):03d}"
        bpp_cls_dir = COMPRESSED_DIR / bpp_tag / cls
        ax = axes[row][i + 1]
        if bpp_cls_dir.exists():
            plot_dct_hist(ax, bpp_cls_dir, f"BPP={bpp}/{cls}", color)
        ax.set_title(f"BPP={bpp} – {cls}")

plt.suptitle("DCT Coefficient Distributions — Real vs Fake under JPEG AI", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dct_histograms.png", dpi=150)
plt.show()

### 7.4 Phase 3 — Mitigation Strategies

**Goal:** Recover detector robustness under JPEG AI compression without full retraining.

We compare two lightweight approaches applied to **ResNet50** (the primary baseline):

| Strategy | Description | Train data | Overhead |
|----------|-------------|------------|----------|
| **MIT-1** — Compression-augmented FT | Add real JPEG AI compressed images (`COMP_TRAIN`, all 6 BPP) to the training set | ORIG_TRAIN + 6×COMP_TRAIN | Higher — dataset grows by 6×`COMP_TRAIN` |
| **MIT-2** — Synthetic JPEG augmentation | Apply standard JPEG (quality 30–95) on-the-fly during training as a proxy for neural compression | ORIG_TRAIN only (online aug, p=0.5) | Lower — same dataset, no extra storage |

**MIT-1** directly exposes the model to actual JPEG AI artefact distributions.
It requires the compressed images to already exist on disk, but the model
learns the real codec's statistical footprint.

**MIT-2** uses standard JPEG as a cheap proxy. Standard JPEG is *not* JPEG AI,
but both suppress high-frequency energy. The question is: is this shared property
sufficient to generalise to JPEG AI artefacts?

**Evaluation** is always on `COMP_TEST` (never seen during any training phase).

We also record **wall-clock training time** to quantify computational overhead —
a key practical consideration when deciding which mitigation to deploy.

In [ ]:

import time

# ── Helper: standard JPEG compression augmentation ───────────────────────────
def _jpeg_compress(img: Image.Image, quality: int) -> Image.Image:
    """Re-encode an image with standard JPEG at a given quality level."""
    arr = np.array(img)
    _, enc = cv2.imencode(".jpg", cv2.cvtColor(arr, cv2.COLOR_RGB2BGR),
                          [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return Image.fromarray(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB))

_jpeg_aug_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    # Apply JPEG compression at a random quality 30–95 with probability 0.5.
    # This is a cheap proxy for neural codec compression: both suppress
    # high-frequency energy, although via different mechanisms.
    T.RandomApply([T.Lambda(lambda img: _jpeg_compress(img, random.randint(30, 95)))], p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def _eval_on_all_sets(model):
    """Evaluate model on ORIG_TEST + COMP_TEST at every BPP (no leakage)."""
    results = {}
    results["original"] = evaluate_detector(model, make_list_loader(ORIG_TEST))
    for bpp in BPP_LEVELS:
        bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
        if bpp_dir.exists():
            results[bpp] = evaluate_detector(model, make_comp_test_loader(bpp))
    return results


# ── Mitigation 1: ORIG_TRAIN + COMP_TRAIN (real JPEG AI compressed images) ───
# The training set is augmented with COMP_TRAIN images — one compressed variant
# per BPP per image. This directly exposes the model to JPEG AI artefacts,
# teaching it to detect fakes even when the neural codec has distorted the
# high-frequency forensic cues.
mit1_ckpt = CHECKPOINTS_DIR / f"{BASELINE_DET}_mit1_augmented.pth"
_mit1_train_time = None

if mit1_ckpt.exists():
    print(f"MIT-1 checkpoint found — skipping training: {mit1_ckpt}")
else:
    aug_samples = list(ORIG_TRAIN)
    for bpp in BPP_LEVELS:
        aug_samples += make_comp_train_samples(bpp)
    print(f"MIT-1 train set: {len(aug_samples)} samples "
          f"({len(ORIG_TRAIN)} orig + {len(aug_samples)-len(ORIG_TRAIN)} compressed)")
    _m1 = load_detector(BASELINE_DET)
    _t0 = time.time()
    _training_histories["MIT-1 (comp. aug)"] = finetune(
        _m1,
        make_list_loader(aug_samples, shuffle=True),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=mit1_ckpt,
        label="mit1",
    )
    _mit1_train_time = time.time() - _t0
    print(f"  MIT-1 training time: {_mit1_train_time/60:.1f} min")

mit1_model = load_detector(BASELINE_DET, pretrained=False)
mit1_model.load_state_dict(torch.load(mit1_ckpt, map_location=DEVICE, weights_only=True))
mit1_model.eval()
print("Evaluating MIT-1 on COMP_TEST...")
mit1_results = _eval_on_all_sets(mit1_model)
for k, v in mit1_results.items():
    print(f"  [MIT1] {k} | AUC={v['auc']:.4f}  acc={v['accuracy']:.4f}")


# ── Mitigation 2: ORIG_TRAIN + synthetic JPEG augmentation ───────────────────
# Instead of adding real compressed images, we apply on-the-fly standard JPEG
# compression during training as a low-cost proxy. No extra disk storage needed.
# The key hypothesis: if the shared property (high-frequency suppression) is the
# main cause of degradation, a different codec (standard JPEG) may still help.
mit2_ckpt = CHECKPOINTS_DIR / f"{BASELINE_DET}_mit2_jpegaug.pth"
_mit2_train_time = None

if mit2_ckpt.exists():
    print(f"\nMIT-2 checkpoint found — skipping training: {mit2_ckpt}")
else:
    print("\nMIT-2: fine-tuning on ORIG_TRAIN with synthetic JPEG augmentation...")
    _m2 = load_detector(BASELINE_DET)
    _t0 = time.time()
    _training_histories["MIT-2 (JPEG aug)"] = finetune(
        _m2,
        make_list_loader(ORIG_TRAIN, shuffle=True, transform=_jpeg_aug_transform),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=mit2_ckpt,
        label="mit2",
    )
    _mit2_train_time = time.time() - _t0
    print(f"  MIT-2 training time: {_mit2_train_time/60:.1f} min")

mit2_model = load_detector(BASELINE_DET, pretrained=False)
mit2_model.load_state_dict(torch.load(mit2_ckpt, map_location=DEVICE, weights_only=True))
mit2_model.eval()
print("Evaluating MIT-2 on COMP_TEST...")
mit2_results = _eval_on_all_sets(mit2_model)
for k, v in mit2_results.items():
    print(f"  [MIT2] {k} | AUC={v['auc']:.4f}  acc={v['accuracy']:.4f}")


# ── Plot training curves (overfitting / convergence check) ───────────────────
if _training_histories:
    plot_training_curves(_training_histories)
else:
    print("All checkpoints loaded from disk — no training history available.")
    print("Delete checkpoints and re-run to see learning curves.")

# ── Save combined CSV (all strategies + both detectors) ──────────────────────
rows = []
for det, strategy, res in [
    (BASELINE_DET, "baseline",       phase1_results),
    (SECOND_DET,   "baseline",       phase1_results_second),
    (BASELINE_DET, "mit1_augmented", mit1_results),
    (BASELINE_DET, "mit2_jpegaug",   mit2_results),
]:
    for k, v in res.items():
        rows.append({"detector": det, "strategy": strategy, "bpp": k,
                     "auc": v["auc"], "accuracy": v["accuracy"], "f1": v["f1"]})
pd.DataFrame(rows).to_csv(RESULTS_DIR / "mitigation_results.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'mitigation_results.csv'}")

# ── Computational overhead ────────────────────────────────────────────────────
_comp_train_n = sum(len(make_comp_train_samples(b)) for b in BPP_LEVELS)
print("\n=== Computational Overhead ===")
if _mit1_train_time:
    print(f"  MIT-1 (comp. aug) : {_mit1_train_time/60:.1f} min"
          f"  |  dataset: {len(ORIG_TRAIN) + _comp_train_n:,} samples")
if _mit2_train_time:
    print(f"  MIT-2 (JPEG aug)  : {_mit2_train_time/60:.1f} min"
          f"  |  dataset: {len(ORIG_TRAIN):,} samples + online augmentation")
if _mit1_train_time and _mit2_train_time:
    print(f"  Overhead ratio MIT-1 / MIT-2: {_mit1_train_time/_mit2_train_time:.2f}×")
elif not _mit1_train_time and not _mit2_train_time:
    print("  (checkpoints loaded from disk — delete them and re-run to measure timing)")


### 7.5 Final Comparison — All Strategies + Both Detectors

This cell consolidates all results into a single degradation plot and a summary table.

**Plot:** AUC vs BPP for all four configurations:
- ResNet50 baseline (no mitigation)
- EfficientNet-B4 baseline (second architecture, no mitigation)
- ResNet50 + MIT-1 (compression-augmented training)
- ResNet50 + MIT-2 (synthetic JPEG augmentation)

**Reading the results:**
- The gap between the "original" AUC (dotted line, not shown in plot) and the BPP curves
  quantifies the degradation caused by JPEG AI.
- The vertical gap between Baseline and MIT-1/MIT-2 at each BPP quantifies the
  **robustness gain** from each mitigation.
- If both baselines (ResNet50 and EfficientNet-B4) show a similar drop pattern,
  the degradation is backbone-agnostic — a general property of JPEG AI forensics disruption.
- The computational overhead table from the previous cell quantifies the cost vs. gain trade-off.

In [ ]:

# Final comparison: baseline (both detectors) vs mitigation strategies (ResNet50)

compare = {
    f"Baseline ({BASELINE_DET})":        {k: v["auc"] for k, v in phase1_results.items() if k != "original"},
    f"Baseline ({SECOND_DET})":          {k: v["auc"] for k, v in phase1_results_second.items() if k != "original"},
    f"MIT-1 comp.aug ({BASELINE_DET})":  {k: v["auc"] for k, v in mit1_results.items()   if k != "original"},
    f"MIT-2 JPEG aug ({BASELINE_DET})":  {k: v["auc"] for k, v in mit2_results.items()   if k != "original"},
}
plot_degradation_curve(compare, metric="auc",
                       title=f"Mitigation Comparison — {BASELINE_DET} + {SECOND_DET}")
print(f"Plot saved → {RESULTS_DIR / 'degradation_auc.png'}")

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for strat, res_dict in [
    (f"Baseline ({BASELINE_DET})",  phase1_results),
    (f"Baseline ({SECOND_DET})",    phase1_results_second),
    ("MIT-1 (comp. aug)",            mit1_results),
    ("MIT-2 (JPEG aug)",             mit2_results),
]:
    for k, v in sorted(res_dict.items(), key=lambda x: (str(x[0]) == "original", x[0])):
        summary_rows.append({"Strategy": strat, "BPP": k,
                              "AUC": f"{v['auc']:.4f}", "Acc": f"{v['accuracy']:.4f}"})
print(pd.DataFrame(summary_rows).to_string(index=False))

# ── Computational overhead ─────────────────────────────────────────────────────
print("\n=== Computational Overhead ===")
if _mit1_train_time:
    print(f"  MIT-1 (comp. aug) : {_mit1_train_time/60:.1f} min")
if _mit2_train_time:
    print(f"  MIT-2 (JPEG aug)  : {_mit2_train_time/60:.1f} min")
if _mit1_train_time and _mit2_train_time:
    print(f"  Overhead ratio MIT-1 / MIT-2: {_mit1_train_time/_mit2_train_time:.2f}×")
elif not _mit1_train_time and not _mit2_train_time:
    print("  (checkpoints loaded from disk — delete them and re-run to measure timing)")
